Python for Stata users
======================

**Author:** Ethan Ligon



## Who this is for



You have used Stata.  You know what a dataset is, what a variable is, what
`collapse` and `merge` and `keep if` do, and probably what `svyset` does.
None of that knowledge is wasted here; most of it transfers directly.

What does *not* transfer is the shape of the language around it.  Stata
gives you one dataset in memory and a vocabulary of commands that act on
it.  Python gives you many named objects, each carrying its own methods,
and you say which object you mean every time.  That is the whole
adjustment, and everything below is a consequence of it.

Twenty minutes, and it runs on the workshop server as it stands.  If you
have never used Stata, read `python_from_scratch.org` instead — it covers
the same ground without assuming the comparison.

-   **What this is not:** a Python course.  It is the shortest path from
    "fluent in Stata" to "able to read the workshop notebooks without
    guessing".



## Setup



In [1]:
# Show tracebacks without the library's internal frames: the line that
# failed, and why.  Change Plain to Verbose if you ever want the rest.
%xmode Plain

import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import lsms_library as ll

print("ready")

Three imports and you have named the tools you will use.  Stata has no
equivalent step because Stata *is* the tool; in Python the language is
general and you say which libraries you want.  `pd` and `ll` are just
shorter names for them.



## 1.  Many objects, not one dataset



In Stata there is a dataset, and it is *the* dataset.  `summarize inc`
means the `inc` in memory.  Load another and the first is gone.

In Python every value has a name you chose and a type that decides what you
can do with it.



In [1]:
region = "Ashanti"
households = 1735
mean_size = 3.28

print(type(region), type(households), type(mean_size))
print(region.upper(), households * 2, round(mean_size, 1))

`str`, `int`, `float`.  The type matters because it decides what is legal:
`region * 2` repeats the text, `households * 2` doubles the number.  Stata
draws the same distinction between string and numeric variables; the
difference is that here *everything* has a type, not just columns.



### Why does my number have a decimal point I did not ask for?



-   **Ask:** I divided two whole numbers and got `3.25` instead of `3`.
-   **Answer:** Division in Python always produces a `float`, even when the
    numbers divide evenly.  If you want the whole-number part, use `//`:
    `13 // 4` is `3`.  This differs from Stata, where integer division is not
    a separate operator.



## 1.  The containers Stata does not have



Stata has variables and, at a stretch, macros.  Python has several ways to
hold more than one thing, and the workshop code uses three of them.



In [1]:
regions = ["Ashanti", "Northern", "Volta"]        # list: ordered, editable
weights = {"Ashanti": 1.84, "Northern": 0.65}     # dict: look up by name
key = ("2016-17", "Ashanti")                      # tuple: fixed, hashable

print(regions[0], "of", len(regions))
print(weights["Ashanti"])
print(key, type(key))

-   **List:** a sequence you can add to and reorder.  Counting starts at **zero**,
    so `regions[0]` is the first element.  Stata's `1` is Python's `0`, and
    this will catch you at least once.
-   **Dict:** a lookup table.  The closest Stata analogue is a value label, but
    a dict is a first-class object you can build and pass around.
-   **Tuple:** a fixed sequence.  It looks like a list with round brackets, and
    you can ignore the difference until section 4, where it turns out that
    the labels identifying a row of survey data *are* tuples.



## 1.  Methods, and why the code is full of full stops



This is the question everyone coming from Stata asks first.

In Stata the verb comes first: `summarize inc`.  In Python the object comes
first and the verb hangs off it: `inc.mean()`.  The full stop means "the
thing on the right belongs to the thing on the left".



In [1]:
name = "Upper West"

print(name.lower())        # a method: belongs to the string
print(len(name))           # a function: stands on its own

counts = [3, 1, 2]
counts.sort()              # a method that CHANGES counts
print(counts)

Because methods return objects that have methods of their own, they chain:
`df.groupby("t").size()` is three steps read left to right — take `df`,
group it, count each group.  Stata would write that as three lines with the
dataset implied.  Python writes it as one line with the object named.



### Why are there so many expressions separated by full stops?



-   **Ask:** I see things like `s.groupby("v")["weight"].mean()`.  Why so many
    full stops?
-   **Answer:** Each full stop asks the object on its left for something.  Read
    it left to right: take `s`, group it by `v`, take the `weight` column of
    that, average it.  Every step hands its result to the next, so a chain is
    just several short commands written on one line.  You can always break it
    up if it is easier to read: assign the middle results to names and take
    one step at a time.



### What is the difference between .shape and .head()?



-   **Ask:** Why does `.shape` have no brackets when `.head()` does?
-   **Answer:** `.shape` is an *attribute* — a fact the object already knows,
    like its number of rows.  `.head()` is a *method* — something the object
    does when you ask, so it needs the brackets that mean "do it now".  If you
    write `df.shape()` you get `TypeError: 'tuple' object is not callable`,
    which is Python saying you tried to run something that was not a verb.



In [1]:
small = pd.DataFrame({"region": ["Ashanti", "Northern"],
                      "weight": [1.84, 0.65]})

print(small.shape)         # attribute: (rows, columns)
print(small.head(1))       # method: do something and give it back

## 1.  Real data, and the index



Now the survey.  The workshop server keeps prepared extracts in your home
directory, and `ll.tools.get_dataframe` reads one.



In [1]:
path = os.path.expanduser("~/extracts/GhanaLSS_household_roster.parquet")
roster = ll.tools.get_dataframe(path)

print("rows and columns:", roster.shape)
print("index names     :", roster.index.names)
roster.head(3)

246,594 rows, eight columns — and four *index* names, `i`, `t`, `v`,
`pid`.  This is the one idea with no Stata counterpart at all, so it is
worth slowing down.

In Stata a row is identified by its position, and if you want to know which
household a row belongs to you keep `hhid` as a variable.  A pandas
DataFrame keeps that identifying information separately, in the **index**.
Here the index is *hierarchical*: household, wave, cluster, person.  The
eight columns are what was measured; the four index levels are who and
when.

A single row's label is therefore four values together — a tuple, which
is why section 2 bothered to mention them.



### Where did my column go?



-   **Ask:** `roster["t"]` raises `KeyError: 't'`, but `t` is clearly there.
-   **Answer:** `t` is an index level, not a column, so it is not in
    `roster.columns`.  Either use `roster.reset_index()` to turn every index
    level into an ordinary column, or select on the index directly with
    `roster.xs(...)`.  For learning, `reset_index()` is the easier habit: it
    gives you a plain rectangle of the kind Stata would hand you.



In [1]:
flat = roster.reset_index()

print(flat.columns.tolist())
print("still", flat.shape[0], "rows")

Twelve columns now: the four index levels have become ordinary columns.
You will see `reset_index()` constantly in the workshop notebooks, and this
is all it does.



## 1.  keep if, and the trap that follows it



Stata's `keep if` has a direct equivalent: build a column of true and false,
and use it to select.



In [1]:
w7 = flat[flat["t"].astype(str) == "2016-17"]
print("2016-17 people:", len(w7))

adults = w7[w7["Age"] >= 18]
print("aged 18 or over:", len(adults))
print("mean age       :", round(float(w7["Age"].mean()), 1))

59,864 people in the 2016–17 round, of whom 31,954 are adults, and a mean
age of 25.0 — a young population, which is the substantive point session
one starts from.

Note what `flat[...]` returned: a **new** DataFrame.  Stata's `keep if`
discards rows from the dataset in memory; the Python expression leaves
`flat` untouched and hands you a separate object.  That is usually what you
want.  It is also the source of the single most expensive mistake a Stata
user makes in pandas.



### I filtered and assigned, and nothing changed



-   **Ask:** I wrote `df[df["a"] > 1]["b"] = 0`, which is what `replace b = 0 if
           a > 1` means, and `b` is unchanged.
-   **Answer:** The first bracket built a new temporary frame, and the
    assignment changed *that*, then threw it away.  Your original `df` never
    saw it.  Do both steps at once with `.loc`, which selects rows and column
    together: `df.loc[df["a"] > 1, "b"] = 0`.  The rule: a selection of a
    selection is a copy, and assigning to a copy changes nothing.  Assignment
    needs a single `.loc`.



In [1]:
toy = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})

toy[toy["a"] > 1]["b"] = 0            # the obvious translation
print("after the obvious version:", toy["b"].tolist())

toy.loc[toy["a"] > 1, "b"] = 0        # what to write instead
print("after .loc            :", toy["b"].tolist())

`[10, 20, 30]` then `[10, 0, 0]`.  The first did nothing and said nothing,
which is what makes it dangerous: no error, no warning you will notice, and
a result that is quietly wrong.  If you take one thing from this tutorial,
take `.loc`.



## 1.  Missing values point the other way



Stata and pandas both have missing, and they disagree about what it means
in a comparison — in opposite directions.



In [1]:
import numpy as np

print("nan > 5   ->", np.nan > 5)
print("nan == nan ->", np.nan == np.nan)
print("missing ages in 2016-17:", int(w7["Age"].isna().sum()))

In Stata, missing sorts *above* every number, so `if age > 60` silently
*includes* the missing.  In pandas, every comparison with missing is False,
so the same filter silently *excludes* them.  Neither raises an error.  Both
change your answer.

The habit worth forming: test for missing explicitly with `.isna()` rather
than relying on a comparison to do it for you.



## 1.  collapse, and why you rarely loop



`collapse (mean) age, by(region)` becomes `groupby`.



In [1]:
print(flat.groupby(flat["t"].astype(str)).size().to_string())

One line per wave, which is the shape `collapse` would have given you.
Note again that `flat` is unchanged: `groupby` returned a new object rather
than replacing the dataset.

Python does have loops, and you will meet `for` in the notebooks, but for
data work you rarely want one.  A Stata user has the right instinct here
already — you would not loop over observations in Stata either, you would
use `by` — and the same applies: operate on whole columns at a time and
let pandas do the iteration.



### Should I write a loop over the rows?



-   **Ask:** How do I loop over households to compute something for each one?
-   **Answer:** Usually you should not.  Whole-column operations
    (`df["Age"].mean()`) and `groupby` are both faster and easier to read than
    a row loop, and they handle missing values properly.  Reach for a loop
    when you are iterating over *files* or *waves* — a handful of things —
    rather than over rows, of which there are a quarter of a million here.



## Where to go next



You now have enough to read the session notebooks without the syntax
getting in the way.  The three things worth keeping in front of you:

-   **Assignment:** use `.loc`, always, whenever you are changing values.
-   **The index:** `reset_index()` turns index levels into columns, and that is
    usually what you want while learning.
-   **Missing:** test with `.isna()`, never with a comparison.

If something breaks, read the last line of the traceback first — the
notebooks are set to show you the failing line and little else — and if it
is still opaque, ask.  Errors in this workshop are data about the
materials, not evidence about you.

